## Chatbots with Message History using LangChain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001EB60C72900>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EB60C734D0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, my name is Vishal Goyal and here I am working as a data scientist and learning AgenticAI.")])

AIMessage(content="Hello Vishal Goyal, nice to meet you. It's great to hear that you're working as a data scientist and exploring AgenticAI. AgenticAI is an interesting field that combines artificial intelligence, cognitive architectures, and agent-based modeling. What specific aspects of AgenticAI are you most interested in learning about, and how do you think it will apply to your work as a data scientist?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 59, 'total_tokens': 141, 'completion_time': 0.2134326, 'prompt_time': 0.010471418, 'queue_time': 0.065618731, 'total_time': 0.223904018}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--a9ceb254-438b-4fcf-94d3-199b65902019-0', usage_metadata={'input_tokens': 59, 'output_tokens': 82, 'total_tokens': 141})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, my name is Vishal Goyal and here I am working as a data scientist and learning AgenticAI."),
        AIMessage(content="Nice to meet you, Vishal Goyal. It's great to hear that you're working as a data scientist and exploring AgenticAI. AgenticAI is a fascinating field that combines artificial intelligence, machine learning, and autonomous systems. What specific aspects of AgenticAI are you interested in or currently learning about? Are you working on any projects or applications that involve AgenticAI? I'm here to help and provide any guidance or support you might need."),
        HumanMessage(content="Hey, what's my name, what I do and learning ?")
    ]
)

AIMessage(content="Your name is Vishal Goyal, you work as a Data Scientist, and you're learning AgenticAI.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 175, 'total_tokens': 199, 'completion_time': 0.045583116, 'prompt_time': 0.017403845, 'queue_time': 0.080481224, 'total_time': 0.062986961}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--4371f7b4-c269-4e37-8629-92aaaf90c358-0', usage_metadata={'input_tokens': 175, 'output_tokens': 24, 'total_tokens': 199})

### Message History
We can use a Message History class to wrap our model and make it stateful.

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [10]:
store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history1=RunnableWithMessageHistory(model,get_session_history)

In [12]:
config1={"configurable":{"session_id":"chat1"}}

In [13]:
response=with_message_history1.invoke(
    [HumanMessage(content="Hi, my name is Vishal Goyal and here I am working as a data scientist and learning AgenticAI.")],
    config=config1
)

In [14]:
response.content

"Hello Vishal Goyal, nice to meet you. It's great that you're a data scientist and taking the initiative to learn about AgenticAI. That's a unique and exciting field that combines AI, autonomy, and decision-making. What drew you to AgenticAI, and what do you hope to achieve or learn from it?"

In [16]:
with_message_history1.invoke(
    [HumanMessage(content="What's my name, and what I Do?")],
    config=config1,
)

AIMessage(content='Your name is Vishal Goyal, and you work as a data scientist, currently learning AgenticAI.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 291, 'total_tokens': 314, 'completion_time': 0.050463044, 'prompt_time': 0.117692488, 'queue_time': 0.057984431, 'total_time': 0.168155532}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--1af4587f-11ba-4e94-97e3-5365e727b304-0', usage_metadata={'input_tokens': 291, 'output_tokens': 23, 'total_tokens': 314})

In [19]:
config2={"configurable":{"session_id":"chat2"}}

response=with_message_history1.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

"I still don't know your name. I'm a large language model, I don't have the ability to recall personal information about you, including your name. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to tell me your name, I'd be happy to chat with you, but I won't be able to guess or recall it."

In [21]:
response=with_message_history1.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Vishal Goyal.'

In [22]:
response=with_message_history1.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config2
)
response.content

"Nice to meet you, John. It's great that you've shared your name with me. However, please keep in mind that I'm a large language model, I don't have the ability to remember your name or any other personal information about you for future conversations. Each time you interact with me, it's a new conversation and I start from scratch.\n\nThat being said, I'm happy to chat with you, John, and help with any questions or topics you'd like to discuss during this conversation. How's your day going so far?"

In [24]:
response=with_message_history1.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

"I remember from our previous conversation, your name is John. However, please keep in mind that this is only for the duration of our current conversation. If you start a new conversation with me in the future, I won't retain any information about your name or our previous conversation.\n\nBut for now, I'm happy to continue chatting with you, John. How can I assist you today?"